# 12 — Recommendation Strategy and Evaluation

## 1. Objective

Notebook 11 trained and compared the machine learning models.

Logistic Regression was selected as the scoring model because it gives strong results while training much faster than Gradient-Boosted Trees.

Its validation predictions were saved in:

`workspace.ml_data.next_basket_lr_validation_predictions`

In this notebook, we will use those saved probabilities to build the final recommendation system.

We will:

- Create the initial Top 5 recommendations
- Evaluate recommendation quality
- Measure End-to-End Recall
- Analyze recommendations for new products
- Test different discovery strategies
- Select the final recommendation strategy
- Save the final recommendation table

No model will be trained again in this notebook.

## 2. Load the Saved Model Predictions

We load the validation predictions created in Notebook 11.

Each row contains:

- A customer
- A candidate product
- The true purchase result
- The predicted purchase probability
- Whether the product is new to the customer

These probabilities will be used to rank products for each customer.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

lr_predictions_df = spark.table(
    "workspace.ml_data.next_basket_lr_validation_predictions"
)

print("Prediction table loaded")

display(
    lr_predictions_df.limit(10)
)

Prediction table loaded


user_id,target_order_id,product_id,target_purchased,is_new_to_customer,candidate_source,purchase_probability
13816,2102595,34,0,0,reorder,0.053036118549291245
144720,1250051,31662,0,0,reorder,0.02757697056182573
192358,3243695,7625,0,0,reorder,0.015308536028536213
46496,3300437,25623,0,0,reorder,0.10116497611856345
105003,2327313,34969,0,0,reorder,0.04809166487708294
155056,2387716,5944,0,0,reorder,0.021744957353172167
168165,871612,20754,0,0,reorder,0.09883397253536408
67643,1181893,17791,0,0,reorder,0.002675427053927426
58612,1544746,37214,0,0,reorder,0.041191910568899925
155056,2387716,27845,0,0,reorder,0.1414185785338229


## 3. Create the Initial Top 5 Recommendations

For each customer, we rank all candidate products by predicted purchase probability.

We then keep the 5 products with the highest probability.

This creates our baseline recommendation list before testing any discovery strategy.

In [0]:
recommendation_window = (
    Window
    .partitionBy("user_id")
    .orderBy(
        F.col("purchase_probability").desc(),
        F.col("product_id").asc()
    )
)

top5_recommendations_df = (
    lr_predictions_df
    .withColumn(
        "recommendation_rank",
        F.row_number().over(recommendation_window)
    )
    .filter(
        F.col("recommendation_rank") <= 5
    )
)

print("Top 5 recommendations created")

display(
    top5_recommendations_df
    .orderBy(
        "user_id",
        "recommendation_rank"
    )
    .limit(20)
)

Top 5 recommendations created


user_id,target_order_id,product_id,target_purchased,is_new_to_customer,candidate_source,purchase_probability,recommendation_rank
14,2316178,29509,1,0,reorder,0.6871980336715175,1
14,2316178,23803,0,0,reorder,0.6460582185396573,2
14,2316178,8744,1,0,reorder,0.5080867030927294,3
14,2316178,37266,0,0,reorder,0.36486852088734056,4
14,2316178,15869,1,0,reorder,0.293682771356237,5
21,1854765,23729,0,0,reorder,0.5683535426858906,1
21,1854765,44632,1,0,reorder,0.3617915822651947,2
21,1854765,28204,0,0,reorder,0.33565992748081386,3
21,1854765,48988,0,0,reorder,0.27522307917062994,4
21,1854765,33894,0,0,reorder,0.24767552364038758,5


## 4. Evaluate the Baseline Top 5 Recommendations

We evaluate the recommendation list using three metrics:

- Precision@5: how many recommended products were actually purchased
- Recall@5: how many of the customer's actual candidate purchases were found in the Top 5
- Hit Rate@5: how often at least one Top 5 recommendation was correct

These metrics evaluate the ranking quality of the recommendation system.

In [0]:
# Number of correct Top 5 recommendations for each customer
top5_user_metrics_df = (
    top5_recommendations_df
    .groupBy("user_id")
    .agg(
        F.sum("target_purchased").alias("correct_recommendations"),
        F.count("*").alias("recommendation_count")
    )
)


# Number of actual purchased products available in the candidate set
actual_purchases_df = (
    lr_predictions_df
    .groupBy("user_id")
    .agg(
        F.sum("target_purchased").alias("actual_purchases")
    )
)


# Combine both
user_metrics_df = (
    top5_user_metrics_df
    .join(
        actual_purchases_df,
        on="user_id",
        how="left"
    )
    .withColumn(
        "precision_at_5",
        F.col("correct_recommendations")
        / F.col("recommendation_count")
    )
    .withColumn(
        "recall_at_5",
        F.when(
            F.col("actual_purchases") > 0,
            F.col("correct_recommendations")
            / F.col("actual_purchases")
        ).otherwise(0.0)
    )
    .withColumn(
        "hit_at_5",
        F.when(
            F.col("correct_recommendations") > 0,
            1.0
        ).otherwise(0.0)
    )
)


# Average across customers
baseline_metrics = (
    user_metrics_df
    .agg(
        F.avg("precision_at_5").alias("precision_at_5"),
        F.avg("recall_at_5").alias("recall_at_5"),
        F.avg("hit_at_5").alias("hit_rate_at_5")
    )
    .first()
)


precision_at_5 = baseline_metrics["precision_at_5"]
recall_at_5 = baseline_metrics["recall_at_5"]
hit_rate_at_5 = baseline_metrics["hit_rate_at_5"]


print("Baseline Top 5 Recommendation Metrics")
print("-------------------------------------")
print("Precision@5:", round(precision_at_5, 4))
print("Recall@5:", round(recall_at_5, 4))
print("Hit Rate@5:", round(hit_rate_at_5, 4))

Baseline Top 5 Recommendation Metrics
-------------------------------------
Precision@5: 0.3793
Recall@5: 0.3558
Hit Rate@5: 0.8025


## 5. Calculate End-to-End Recall@5

The previous Recall@5 only considers purchased products that were included in our candidate set.

However, the recommendation system cannot recommend a product that was never generated as a candidate.

To evaluate the complete system, we calculate End-to-End Recall@5.

This compares the correct Top 5 recommendations with all products that the customer actually purchased in the target order.

In [0]:
# Target orders in the validation set
validation_target_orders_df = (
    lr_predictions_df
    .select(
        "user_id",
        "target_order_id"
    )
    .distinct()
)


# Load the complete order-product data
order_products_df = spark.table(
    "workspace.cleaned_data.order_products"
)


# Count every product actually purchased in each target order
actual_full_basket_df = (
    validation_target_orders_df.alias("v")
    .join(
        order_products_df.alias("o"),
        F.col("v.target_order_id") == F.col("o.order_id"),
        how="inner"
    )
    .groupBy(
        F.col("v.user_id").alias("user_id")
    )
    .agg(
        F.countDistinct(
            F.col("o.product_id")
        ).alias("actual_full_basket_size")
    )
)


# Number of correct Top 5 recommendations
correct_top5_df = (
    top5_recommendations_df
    .groupBy("user_id")
    .agg(
        F.sum("target_purchased").alias(
            "correct_top5"
        )
    )
)


# Calculate End-to-End Recall for each customer
end_to_end_user_metrics_df = (
    actual_full_basket_df
    .join(
        correct_top5_df,
        on="user_id",
        how="left"
    )
    .fillna(
        0,
        subset=["correct_top5"]
    )
    .withColumn(
        "end_to_end_recall_at_5",
        F.col("correct_top5")
        / F.col("actual_full_basket_size")
    )
)


# Average across customers
end_to_end_recall_at_5 = (
    end_to_end_user_metrics_df
    .agg(
        F.avg(
            "end_to_end_recall_at_5"
        ).alias("end_to_end_recall_at_5")
    )
    .first()["end_to_end_recall_at_5"]
)


print(
    "End-to-End Recall@5:",
    round(end_to_end_recall_at_5, 4)
)

End-to-End Recall@5: 0.2492


## 6. Evaluate New-Product Discovery

Candidate generation allows the system to recommend products the customer has never purchased before.

We now measure how well the baseline Top 5 recommendations discover these new products.

In [0]:
# New products that were actually purchased
actual_new_df = (
    lr_predictions_df
    .filter(
        (F.col("is_new_to_customer") == 1) &
        (F.col("target_purchased") == 1)
    )
)

actual_new_products = actual_new_df.count()

customers_with_actual_new = (
    actual_new_df
    .select("user_id")
    .distinct()
    .count()
)


# New products recommended in the baseline Top 5
recommended_new_df = (
    top5_recommendations_df
    .filter(F.col("is_new_to_customer") == 1)
)

new_products_recommended = recommended_new_df.count()

correct_new_products = (
    recommended_new_df
    .filter(F.col("target_purchased") == 1)
    .count()
)

customers_with_correct_new = (
    recommended_new_df
    .filter(F.col("target_purchased") == 1)
    .select("user_id")
    .distinct()
    .count()
)


# Metrics
new_product_precision = (
    correct_new_products / new_products_recommended
    if new_products_recommended > 0 else 0
)

new_product_recall = (
    correct_new_products / actual_new_products
    if actual_new_products > 0 else 0
)

new_product_hit_rate = (
    customers_with_correct_new / customers_with_actual_new
    if customers_with_actual_new > 0 else 0
)


print("Baseline New-Product Discovery")
print("--------------------------------")
print("Actual new products purchased:", actual_new_products)
print("New products recommended:", new_products_recommended)
print("Correct new recommendations:", correct_new_products)
print("New-product Precision:", round(new_product_precision, 4))
print("New-product Recall:", round(new_product_recall, 4))
print("New-product Hit Rate:", round(new_product_hit_rate, 4))

Baseline New-Product Discovery
--------------------------------
Actual new products purchased: 9081
New products recommended: 893
Correct new recommendations: 20
New-product Precision: 0.0224
New-product Recall: 0.0022
New-product Hit Rate: 0.0031


## 7. Test Discovery Strategies

The baseline mainly recommends products the customer already knows.

We compare it with:

- A forced strategy: 4 known products + 1 new product
- Conditional discovery: introduce a new product only when its predicted probability is high enough

We test probability thresholds of 0.02, 0.03, and 0.04.

In [0]:
# Windows for ranking known and new products
known_window = (
    Window
    .partitionBy("user_id")
    .orderBy(
        F.col("purchase_probability").desc(),
        F.col("product_id").asc()
    )
)

new_window = known_window


# --------------------------------------------------
# 1. Forced strategy: 4 known + 1 new product
# --------------------------------------------------

top4_known_df = (
    lr_predictions_df
    .filter(F.col("is_new_to_customer") == 0)
    .withColumn(
        "rank",
        F.row_number().over(known_window)
    )
    .filter(F.col("rank") <= 4)
    .drop("rank")
)

best_new_df = (
    lr_predictions_df
    .filter(F.col("is_new_to_customer") == 1)
    .withColumn(
        "new_rank",
        F.row_number().over(new_window)
    )
    .filter(F.col("new_rank") == 1)
    .drop("new_rank")
)

forced_hybrid_df = (
    top4_known_df
    .unionByName(best_new_df)
)


# --------------------------------------------------
# 2. Conditional discovery
# --------------------------------------------------

# Customers who already have a new product in baseline Top 5
baseline_has_new_df = (
    top5_recommendations_df
    .filter(F.col("is_new_to_customer") == 1)
    .select("user_id")
    .distinct()
)


def create_conditional_strategy(threshold):

    # Best new product above threshold
    qualified_new_df = (
        best_new_df
        .filter(
            F.col("purchase_probability") >= threshold
        )
    )

    # Only replace a product when baseline has no new product
    replacement_users_df = (
        qualified_new_df
        .select("user_id")
        .join(
            baseline_has_new_df,
            on="user_id",
            how="left_anti"
        )
    )

    # Keep first 4 baseline products for these customers
    replacement_baseline_df = (
        top5_recommendations_df
        .join(
            replacement_users_df,
            on="user_id",
            how="inner"
        )
        .filter(F.col("recommendation_rank") <= 4)
        .drop("recommendation_rank")
    )

    # Add their best new product
    replacement_new_df = (
        qualified_new_df
        .join(
            replacement_users_df,
            on="user_id",
            how="inner"
        )
    )

    # Customers who do not need replacement keep baseline Top 5
    unchanged_df = (
        top5_recommendations_df
        .join(
            replacement_users_df,
            on="user_id",
            how="left_anti"
        )
        .drop("recommendation_rank")
    )

    return (
        unchanged_df
        .unionByName(replacement_baseline_df)
        .unionByName(replacement_new_df)
    )


# --------------------------------------------------
# 3. Evaluation function
# --------------------------------------------------

def evaluate_strategy(name, recommendations_df):

    customer_results = (
        recommendations_df
        .groupBy("user_id")
        .agg(
            F.sum("target_purchased").alias("correct"),
            F.count("*").alias("recommended")
        )
        .join(
            actual_purchases_df,
            on="user_id",
            how="left"
        )
        .withColumn(
            "precision",
            F.col("correct") / F.col("recommended")
        )
        .withColumn(
            "recall",
            F.when(
                F.col("actual_purchases") > 0,
                F.col("correct") / F.col("actual_purchases")
            ).otherwise(0.0)
        )
        .withColumn(
            "hit",
            F.when(F.col("correct") > 0, 1.0).otherwise(0.0)
        )
    )

    metrics = (
        customer_results
        .agg(
            F.avg("precision").alias("precision_at_5"),
            F.avg("recall").alias("recall_at_5"),
            F.avg("hit").alias("hit_rate_at_5")
        )
        .first()
    )

    new_stats = (
        recommendations_df
        .filter(F.col("is_new_to_customer") == 1)
        .agg(
            F.count("*").alias("new_recommended"),
            F.sum("target_purchased").alias("correct_new")
        )
        .first()
    )

    return (
        name,
        metrics["precision_at_5"],
        metrics["recall_at_5"],
        metrics["hit_rate_at_5"],
        new_stats["new_recommended"],
        new_stats["correct_new"]
    )


# --------------------------------------------------
# 4. Compare all strategies
# --------------------------------------------------

strategy_002_df = create_conditional_strategy(0.02)
strategy_003_df = create_conditional_strategy(0.03)
strategy_004_df = create_conditional_strategy(0.04)

results = [
    evaluate_strategy(
        "Baseline",
        top5_recommendations_df
    ),
    evaluate_strategy(
        "Forced 4+1",
        forced_hybrid_df
    ),
    evaluate_strategy(
        "Conditional >= 0.02",
        strategy_002_df
    ),
    evaluate_strategy(
        "Conditional >= 0.03",
        strategy_003_df
    ),
    evaluate_strategy(
        "Conditional >= 0.04",
        strategy_004_df
    )
]

strategy_comparison_df = spark.createDataFrame(
    results,
    [
        "strategy",
        "precision_at_5",
        "recall_at_5",
        "hit_rate_at_5",
        "new_products_recommended",
        "correct_new_products"
    ]
)

display(strategy_comparison_df)

strategy,precision_at_5,recall_at_5,hit_rate_at_5,new_products_recommended,correct_new_products
Baseline,0.3793474962063778,0.35581607291515377,0.8025037936267071,893,20
Forced 4+1,0.33036545270612505,0.317929438453458,0.7809180576631259,26360,821
Conditional >= 0.02,0.33691198786039916,0.32314345636377323,0.7844081942336874,23385,798
Conditional >= 0.03,0.358391502276181,0.34053346081102603,0.7948406676783004,11321,453
Conditional >= 0.04,0.37632018209408696,0.3543153060223505,0.801669195751138,1988,75


## 8. Select and Save the Final Strategy

The conditional strategy with a probability threshold of 0.04 gives the best balance between accuracy and product discovery.

It increases the number of useful new-product recommendations while keeping Precision@5, Recall@5, and Hit Rate@5 close to the baseline.

We therefore select this strategy as the final recommendation strategy.

In [0]:
# Select final strategy
final_recommendations_df = (
    strategy_004_df
    .withColumn(
        "recommendation_strategy",
        F.lit("conditional_hybrid_0.04")
    )
)


# Recalculate final ranking
final_window = (
    Window
    .partitionBy("user_id")
    .orderBy(
        F.col("purchase_probability").desc(),
        F.col("product_id").asc()
    )
)

final_recommendations_df = (
    final_recommendations_df
    .withColumn(
        "recommendation_rank",
        F.row_number().over(final_window)
    )
)


# Save final recommendations
final_table = (
    "workspace.ml_data."
    "next_basket_final_recommendations"
)

(
    final_recommendations_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(final_table)
)


print("Final strategy: Conditional >= 0.04")
print("Final recommendation table:", final_table)
print("Recommendations:", final_recommendations_df.count())

Final strategy: Conditional >= 0.04
Final recommendation table: workspace.ml_data.next_basket_final_recommendations
Recommendations: 131800


## 9. Create the Final Serving Table

The saved recommendation table contains product IDs.

For easier use in dashboards or applications, we add the product, aisle, and department names and save a clean serving table.

In [0]:
# Load product information
products_df = spark.table(
    "workspace.cleaned_data.products"
)

aisles_df = spark.table(
    "workspace.cleaned_data.aisles"
)

departments_df = spark.table(
    "workspace.cleaned_data.departments"
)


# Add readable product information
serving_recommendations_df = (
    final_recommendations_df.alias("r")
    .join(
        products_df.alias("p"),
        F.col("r.product_id") == F.col("p.product_id"),
        "left"
    )
    .join(
        aisles_df.alias("a"),
        F.col("p.aisle_id") == F.col("a.aisle_id"),
        "left"
    )
    .join(
        departments_df.alias("d"),
        F.col("p.department_id") == F.col("d.department_id"),
        "left"
    )
    .select(
        F.col("r.user_id"),
        F.col("r.target_order_id"),
        F.col("r.recommendation_rank"),
        F.col("r.product_id"),
        F.col("p.product_name"),
        F.col("a.aisle").alias("aisle"),
        F.col("d.department").alias("department"),
        F.col("r.purchase_probability"),
        F.col("r.is_new_to_customer"),
        F.col("r.candidate_source"),
        F.col("r.recommendation_strategy")
    )
)


# Save serving table
serving_table = (
    "workspace.ml_data."
    "next_basket_serving_recommendations"
)

(
    serving_recommendations_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(serving_table)
)


print("Serving table saved:", serving_table)
print("Rows:", serving_recommendations_df.count())

display(
    serving_recommendations_df
    .orderBy("user_id", "recommendation_rank")
    .limit(15)
)

Serving table saved: workspace.ml_data.next_basket_serving_recommendations
Rows: 131800


user_id,target_order_id,recommendation_rank,product_id,product_name,aisle,department,purchase_probability,is_new_to_customer,candidate_source,recommendation_strategy
14,2316178,1,29509,80 Vodka Holiday Edition,spirits,alcohol,0.6871980336715175,0,reorder,conditional_hybrid_0.04
14,2316178,2,23803,Jalapeno Pepper,fresh vegetables,produce,0.6460582185396573,0,reorder,conditional_hybrid_0.04
14,2316178,3,8744,Mixed Vegetables,frozen produce,frozen,0.5080867030927294,0,reorder,conditional_hybrid_0.04
14,2316178,4,37266,Tater Treats Seasoned Shredded Potatoes,frozen appetizers sides,frozen,0.36486852088734056,0,reorder,conditional_hybrid_0.04
14,2316178,5,15869,Sweet Hot Dog Buns,buns rolls,bakery,0.293682771356237,0,reorder,conditional_hybrid_0.04
21,1854765,1,23729,Hard Boiled Eggs,eggs,dairy eggs,0.5683535426858906,0,reorder,conditional_hybrid_0.04
21,1854765,2,44632,Sparkling Water Grapefruit,water seltzer sparkling water,beverages,0.3617915822651947,0,reorder,conditional_hybrid_0.04
21,1854765,3,28204,Organic Fuji Apple,fresh fruits,produce,0.33565992748081386,0,reorder,conditional_hybrid_0.04
21,1854765,4,48988,Unsweetened Premium Iced Tea,tea,beverages,0.27522307917062994,0,reorder,conditional_hybrid_0.04
21,1854765,5,33894,Goldfish Cheddar Baked Snack Crackers Multi Packs,crackers,snacks,0.24767552364038758,0,reorder,conditional_hybrid_0.04


## 10. Conclusion

The final recommendation system uses Logistic Regression scores with a conditional discovery strategy.

A new product is introduced only when its predicted purchase probability is at least 0.04.

Compared with the baseline, this strategy:

- Keeps recommendation accuracy almost unchanged
- Increases new-product recommendations from 893 to 1,988
- Increases correct new-product recommendations from 20 to 75

The final recommendations are saved in:

`workspace.ml_data.next_basket_final_recommendations`

A human-readable serving version is saved in:

`workspace.ml_data.next_basket_serving_recommendations`

This table is ready to be used by a dashboard, application, or external serving database.